# Spindle — Quick Start

**Spindle** generates statistically realistic, relationally correct synthetic data for Microsoft Fabric.

This notebook covers:
- Installing and importing Spindle
- Generating the Retail domain at `fabric_demo` scale
- Exploring the resulting DataFrames
- Verifying referential integrity
- Overriding distributions at runtime

In [1]:
%pip install sqllocks-spindle --quiet


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /Users/sqllocks/Library/CloudStorage/OneDrive-Personal/VSCode/AzureClients/forge-workspace/projects/fabric-datagen/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Generate retail data

The `fabric_demo` scale preset generates ~200 customers and ~1,000 orders — fast and ideal for notebooks.

In [2]:
from sqllocks_spindle import Spindle, RetailDomain

spindle = Spindle()
result = spindle.generate(domain=RetailDomain(), scale="fabric_demo", seed=42)
print(result)

  Generating customer... (200 rows)
  Generating address... (300 rows)
  Generating product_category... (50 rows)
  Generating product... (100 rows)
  Generating promotion... (200 rows)
  Generating store... (150 rows)
  Generating order... (1,000 rows)
  Generating order_line... (2,500 rows)
  Generating return... (170 rows)
GenerationResult(9 tables, 4,670 total rows, 0.1s)


In [3]:
print(result.summary())

Spindle Generation Result
Schema: retail_3nf
Domain: retail
Mode:   3nf
Seed:   42
Time:   0.1s

Table                             Rows  Columns
---------------------------------------------
customer                           200        8
address                            300       10
product_category                    50        4
product                            100        6
promotion                          200        6
store                              150        5
order                            1,000        8
order_line                       2,500        8
return                             170        5
---------------------------------------------
TOTAL                            4,670


## Explore the tables

In [4]:
result["customer"].head()

,customer_id,first_name,last_name,email,gender,loyalty_tier,signup_date,is_active
0,1,Dillon,Macdonald,hyoung@example.org,F,Silver,2025-08-11 20:36:31.081033514,true
1,2,Andrew,Paul,brianna69@example.com,M,Basic,2024-11-18 05:11:09.502839751,true
2,3,Jasmine,Schaefer,lsmith@example.net,M,Platinum,2025-11-07 06:55:16.583412953,true
3,4,John,Evans,walvarez@example.net,M,Gold,2025-02-15 17:27:40.257596628,true
4,5,Andrea,Ramos,teresa60@example.org,M,Basic,2025-06-19 12:43:20.628112464,true


In [5]:
result["order"].head()

,order_id,customer_id,store_id,shipping_address_id,promotion_id,order_date,status,order_total
0,1,1,101,254,143,2025-08-12 20:36:31.081033514,completed,156.40
1,2,10,46,None,None,2025-08-19 12:01:03.000000000,completed,76.20
2,3,24,1,264,None,2025-03-26 07:29:42.000000000,shipped,57.96
3,4,5,1,256,None,2025-06-20 12:43:20.628112464,completed,0.00
4,5,131,113,253,None,2024-04-25 02:52:33.000000000,shipped,263.71


## Address data — lat/lng ready for Power BI maps

In [6]:
result["address"][["city", "state", "zip_code", "lat", "lng"]].head(10)

,city,state,zip_code,lat,lng
0,Port Henry,NY,12974,44.0465,-73.4705
1,Medical Lake,WA,99022,47.6150,-117.7040
2,Parachute,CO,81635,39.4519,-108.0529
3,Berea,KY,40404,37.5687,-84.2963
4,Fall River Mills,CA,96028,41.0393,-121.4606
5,Beckwourth,CA,96129,39.7721,-120.4051
6,Peculiar,MO,64078,38.7165,-94.4405
7,Imogene,IA,51645,40.8631,-95.4358
8,Bucksport,ME,04416,44.6015,-68.7768
9,Presque Isle,MI,49777,45.2969,-83.5023


## Pareto order distribution

Spindle uses a Pareto distribution for customer orders — 20% of customers place ~80% of orders.

In [7]:
order_counts = result["order"].groupby("customer_id").size().sort_values(ascending=False)
print(f"Max orders per customer:    {order_counts.max()}")
print(f"Median orders per customer: {order_counts.median():.1f}")
print(f"\nTop customers by order count:")
print(order_counts.head(10))

Max orders per customer:    50
Median orders per customer: 3.0

Top customers by order count:
customer_id
1     50
2     50
3     50
4     50
5     50
6     50
7     35
10    28
8     25
12    25
dtype: int64


## Referential integrity

In [8]:
errors = result.verify_integrity()
if errors:
    for e in errors:
        print(f"FK error: {e}")
else:
    print("Referential integrity: PASS — no FK violations")

Referential integrity: PASS — no FK violations


## Distribution check — loyalty tiers

In [9]:
result["customer"]["loyalty_tier"].value_counts(normalize=True).mul(100).round(1).rename("pct %")

loyalty_tier
Basic       54.0
Silver      27.0
Gold        13.5
Platinum     5.5
Name: pct %, dtype: float64

## Override distributions at runtime

Pass `overrides={}` to simulate different business scenarios without editing any config files.

In [10]:
# Simulate a premium-skewed customer base (e.g. loyalty program re-launch)
premium_domain = RetailDomain(overrides={
    "customer.loyalty_tier": {"Basic": 0.15, "Silver": 0.25, "Gold": 0.35, "Platinum": 0.25},
})

r2 = spindle.generate(domain=premium_domain, scale="fabric_demo", seed=42)
r2["customer"]["loyalty_tier"].value_counts(normalize=True).mul(100).round(1).rename("pct %")

  Generating customer... (200 rows)
  Generating address... (300 rows)
  Generating product_category... (50 rows)
  Generating product... (100 rows)
  Generating promotion... (200 rows)
  Generating store... (150 rows)
  Generating order... (1,000 rows)
  Generating order_line... (2,500 rows)
  Generating return... (170 rows)


loyalty_tier
Gold        36.0
Platinum    24.5
Silver      23.0
Basic       16.5
Name: pct %, dtype: float64

In [11]:
# Reproducibility — same seed always gives identical output
r3 = spindle.generate(domain=RetailDomain(), scale="fabric_demo", seed=42)
r4 = spindle.generate(domain=RetailDomain(), scale="fabric_demo", seed=42)
assert list(r3["customer"]["customer_id"]) == list(r4["customer"]["customer_id"])
print("Same seed → identical output: PASS")

  Generating customer... (200 rows)


  Generating address... (300 rows)
  Generating product_category... (50 rows)
  Generating product... (100 rows)
  Generating promotion... (200 rows)
  Generating store... (150 rows)
  Generating order... (1,000 rows)
  Generating order_line... (2,500 rows)
  Generating return... (170 rows)
  Generating customer... (200 rows)
  Generating address... (300 rows)
  Generating product_category... (50 rows)
  Generating product... (100 rows)
  Generating promotion... (200 rows)
  Generating store... (150 rows)
  Generating order... (1,000 rows)
  Generating order_line... (2,500 rows)
  Generating return... (170 rows)
Same seed → identical output: PASS
